# Bases de données — Exercices 4 et 5 : clés primaires (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue : la requête SQL à taper dans la **console H2**, et le résultat obtenu |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** identifiant (clé), clés candidates, clé primaire, unicité des enregistrements, contrainte de clé primaire.

Ces deux exercices sont des questions de **réflexion** : la réponse est un texte justifié, comme à l'écrit.

> ⚠️ **Liberté prise :** pour **illustrer** les réponses, on a ajouté des petites requêtes à essayer dans la console H2, avec des données inventées (élèves et trajets imaginaires). Elles ne sont pas demandées par le sujet.

> **Rappel du cours (définition).** Un **identifiant** (appelé aussi **clé**) d'une relation est la donnée d'un ou plusieurs attributs qui permet d'identifier de manière unique un enregistrement. En d'autres termes, deux enregistrements ne peuvent avoir le même identifiant.
> Dans une relation, il y a en général plusieurs choix possibles pour un identifiant (appelés **clés candidates**). La donnée d'un identifiant d'une relation est appelée **clé primaire**.

---
# MISE EN PLACE

En classe, les requêtes SQL sont tapées dans la **console H2** (le SGBD utilisé dans le cours). C'est aussi le cas ici : **chaque requête de ce notebook a été exécutée dans H2**, et le résultat affiché sous la requête est celui qu'a donné H2. Il n'y a **aucun code à exécuter** dans ce notebook : il suffit de recopier les requêtes dans la console H2.

**Pour ouvrir la console H2 :** lancer « H2 Console », se connecter avec la JDBC URL `jdbc:h2:~/nsi`, l'utilisateur `sa` et un mot de passe vide.

> ⚠️ **Liberté prise :** les résultats ont été obtenus avec H2 version 2.5.252 et recopiés tels quels, à trois détails près : la durée d'exécution affichée par H2 (« 12 ms ») a été supprimée, car elle change à chaque fois ; pour les résultats longs, seules les 12 premières lignes sont recopiées, suivies de « ... » et du nombre total de lignes ; dans les messages d'erreur, le nom technique de l'erreur Java (`org.h2.jdbc....Exception`) a été remplacé par « Erreur : » ; quand plusieurs instructions sont exécutées d'un coup, chacune est rappelée au-dessus de son résultat. Dans la console du navigateur, le résultat apparaît sous forme de tableau ; le contenu est le même.

---
# EXERCICE 4

#### 📝 ÉNONCÉ

> On considère une relation regroupant des informations sur les élèves d'un lycée, dont les attributs sont les suivants : nom, prénom, date de naissance, adresse, matricule.
> Parmi les données ci-dessous, lesquelles peuvent être utilisées comme clé primaire ? Justifier.
>
> - nom ;
> - adresse ;
> - (nom, prénom) ;
> - (nom, prénom, adresse) ;
> - matricule ;
> - (nom, matricule).

#### ✅ RÉSOLUTION

La question à se poser pour chaque proposition : **peut-il exister deux élèves différents avec la même valeur ?** Si oui, ce n'est pas une clé.

| Proposition | Clé primaire ? | Justification |
|:---|:---:|:---|
| nom | ❌ | Deux élèves peuvent avoir le même nom (deux frères et sœurs, deux « Martin »). |
| adresse | ❌ | Des frères et sœurs, ou des élèves d'un même immeuble, partagent la même adresse. |
| (nom, prénom) | ❌ | Les homonymes existent : deux « Léa Martin » peuvent être dans le même lycée. |
| (nom, prénom, adresse) | ❌ | C'est rare, mais **pas garanti** : par exemple deux cousins homonymes hébergés à la même adresse. De plus, une adresse change quand on déménage, alors qu'un identifiant doit rester stable. |
| **matricule** | ✅ | Le matricule est **attribué** par l'établissement justement pour être **unique** : deux élèves n'ont jamais le même. C'est **le bon choix** de clé primaire. |
| (nom, matricule) | ✅ (mais à éviter) | Deux élèves ne peuvent pas avoir le même couple (nom, matricule), puisque le matricule suffit déjà à les distinguer. Mais le nom est **inutile** : on choisit une clé **minimale**, donc `matricule` seul. |

**Réponse :** peuvent servir de clé primaire `matricule` et `(nom, matricule)` ; on choisit `matricule`, car il suffit à lui seul.

*Illustration (non demandée) : avec (nom, prénom) comme clé primaire, deux élèves homonymes. La deuxième insertion est refusée par H2.*

```sql
CREATE TABLE Eleves1 (
    nom VARCHAR, prenom VARCHAR, naissance DATE, adresse VARCHAR, matricule INT,
    PRIMARY KEY (nom, prenom) );
INSERT INTO Eleves1 VALUES ('Martin', 'Léa', '2008-03-12', '3 rue Hoche', 1001);
INSERT INTO Eleves1 VALUES ('Martin', 'Léa', '2009-07-01', '8 avenue Foch', 1002);
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Eleves1 ( nom VARCHAR, prenom VARCHAR, naissance DATE, adresse VARCHAR, matricule INT, PRIMARY KEY (nom, prenom) );
(Update count: 0)

INSERT INTO Eleves1 VALUES ('Martin', 'Léa', '2008-03-12', '3 rue Hoche', 1001);
(Update count: 1)

INSERT INTO Eleves1 VALUES ('Martin', 'Léa', '2009-07-01', '8 avenue Foch', 1002);
Erreur : Violation d'index unique ou clé primaire: "PUBLIC.CONSTRAINT_C INDEX PUBLIC.PRIMARY_KEY_C ON PUBLIC.ELEVES1(NOM, PRENOM) VALUES ( /* key:1 */ 'Martin', U&'L\\00e9a')"
Unique index or primary key violation: "PUBLIC.CONSTRAINT_C INDEX PUBLIC.PRIMARY_KEY_C ON PUBLIC.ELEVES1(NOM, PRENOM) VALUES ( /* key:1 */ 'Martin', U&'L\\00e9a')"; SQL statement:
INSERT INTO Eleves1 VALUES ('Martin', 'Léa', '2009-07-01', '8 avenue Foch', 1002) [23505-252]
```

*Avec matricule comme clé primaire, les deux élèves sont acceptés :*

```sql
CREATE TABLE Eleves2 (
    nom VARCHAR, prenom VARCHAR, naissance DATE, adresse VARCHAR, matricule INT,
    PRIMARY KEY (matricule) );
INSERT INTO Eleves2 VALUES ('Martin', 'Léa', '2008-03-12', '3 rue Hoche', 1001);
INSERT INTO Eleves2 VALUES ('Martin', 'Léa', '2009-07-01', '8 avenue Foch', 1002);
SELECT * FROM Eleves2;
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Eleves2 ( nom VARCHAR, prenom VARCHAR, naissance DATE, adresse VARCHAR, matricule INT, PRIMARY KEY (matricule) );
(Update count: 0)

INSERT INTO Eleves2 VALUES ('Martin', 'Léa', '2008-03-12', '3 rue Hoche', 1001);
(Update count: 1)

INSERT INTO Eleves2 VALUES ('Martin', 'Léa', '2009-07-01', '8 avenue Foch', 1002);
(Update count: 1)

SELECT * FROM Eleves2;
NOM    | PRENOM | NAISSANCE  | ADRESSE       | MATRICULE
Martin | Léa    | 2008-03-12 | 3 rue Hoche   | 1001
Martin | Léa    | 2009-07-01 | 8 avenue Foch | 1002
(2 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Le test d'une clé :** une clé doit être unique **pour toutes les données possibles**, pas seulement pour les données actuelles. « Il n'y a pas d'homonymes cette année » ne suffit pas : il pourrait y en avoir l'an prochain.
- **Pourquoi des identifiants artificiels ?** Dans la vie réelle, presque aucune caractéristique naturelle n'est unique (noms, adresses, dates de naissance). C'est pour cela qu'on **invente** des numéros : matricule, numéro de sécurité sociale, numéro de commande, `id` dans les tables de l'exercice 2...
- **Clés candidates :** `matricule` et `(nom, matricule)` sont toutes deux des identifiants. On choisit comme **clé primaire** le plus simple (minimal).
- **Ce que fait le SGBD :** une fois la clé primaire déclarée, H2 **refuse** toute insertion qui créerait un doublon de clé : c'est la *contrainte de clé primaire* (message « Violation d'index unique ou clé primaire »). Dans l'illustration, avec la clé `(nom, prénom)`, la deuxième Léa Martin ne peut pas être inscrite : c'est bien la preuve que ce choix de clé est mauvais.

---
# EXERCICE 5

#### 📝 ÉNONCÉ

> Une relation contient des informations sur les trajets réalisés par les utilisateurs d'un service de location de vélos. Les attributs de cette relation sont les suivants : date, heures de départ et d'arrivée, stations de départ et d'arrivée.
> L'unicité des enregistrements est-elle garantie ? Proposer une solution afin de pouvoir définir une clé primaire.

#### ✅ RÉSOLUTION

**L'unicité n'est pas garantie.** Deux utilisateurs différents peuvent emprunter un vélo **à la même station, à la même heure**, et le rendre **à la même station, à la même heure** (deux amis qui font le trajet ensemble, par exemple). On aurait alors deux enregistrements identiques : aucun attribut, ni aucun groupe d'attributs, ne permet de les distinguer.

**Solution :** ajouter un attribut qui sert d'identifiant, par exemple :
- un **numéro de trajet** `id_trajet`, attribué par le service et différent pour chaque trajet (comme l'attribut `id` des tables de l'exercice 2). Il devient la clé primaire :

  Trajets(<u>id_trajet</u>, date, heure_depart, heure_arrivee, station_depart, station_arrivee)

- ou bien un **identifiant de l'utilisateur** `id_utilisateur` : comme un utilisateur ne peut pas commencer deux trajets au même instant, le groupe (`id_utilisateur`, `date`, `heure_depart`) identifie un trajet. Cela suppose cependant qu'un utilisateur ne loue qu'un vélo à la fois.

La première solution est la plus simple et la plus sûre.

*Illustration (non demandée) : deux amis font le même trajet. Grâce à `id_trajet`, les deux trajets sont distincts.*

```sql
CREATE TABLE Trajets (
    id_trajet INT, date_trajet DATE, heure_depart TIME, heure_arrivee TIME,
    station_depart VARCHAR, station_arrivee VARCHAR,
    PRIMARY KEY (id_trajet) );
INSERT INTO Trajets VALUES (1, '2026-11-14', '08:05:00', '08:20:00', 'Lecourbe', 'Bastille');
INSERT INTO Trajets VALUES (2, '2026-11-14', '08:05:00', '08:20:00', 'Lecourbe', 'Bastille');
SELECT * FROM Trajets;
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Trajets ( id_trajet INT, date_trajet DATE, heure_depart TIME, heure_arrivee TIME, station_depart VARCHAR, station_arrivee VARCHAR, PRIMARY KEY (id_trajet) );
(Update count: 0)

INSERT INTO Trajets VALUES (1, '2026-11-14', '08:05:00', '08:20:00', 'Lecourbe', 'Bastille');
(Update count: 1)

INSERT INTO Trajets VALUES (2, '2026-11-14', '08:05:00', '08:20:00', 'Lecourbe', 'Bastille');
(Update count: 1)

SELECT * FROM Trajets;
ID_TRAJET | DATE_TRAJET | HEURE_DEPART | HEURE_ARRIVEE | STATION_DEPART | STATION_ARRIVEE
1         | 2026-11-14  | 08:05:00     | 08:20:00      | Lecourbe       | Bastille
2         | 2026-11-14  | 08:05:00     | 08:20:00      | Lecourbe       | Bastille
(2 rows)
```

> ⚠️ **Liberté prise :** dans l'illustration, les attributs s'appellent `date_trajet`, `heure_depart`... au lieu de `date`, `heure` : `DATE` et `TIME` sont des **mots réservés** de SQL (des noms de types), qu'il vaut mieux éviter comme noms d'attributs.

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Méthode :** pour montrer que l'unicité n'est **pas** garantie, il suffit de trouver **un exemple concret** de deux enregistrements différents (deux vrais trajets) qui auraient exactement les mêmes valeurs. Un seul contre-exemple suffit.
- **Rédaction :** une bonne réponse donne le contre-exemple **et** propose une solution précise (le nom de l'attribut ajouté, et le schéma de la relation avec la clé soulignée).
- Dans l'illustration, les deux enregistrements ne diffèrent **que** par `id_trajet` : sans cet attribut, ce seraient deux lignes identiques, impossibles à distinguer (et à modifier ou supprimer séparément).